# Foundation Model Transparency Index: scores by developer and indicator

**Purpose.** This notebook builds `fmti_dec2025_scores_tidy.csv` and `fmti_companies.csv`, the two tables behind View 1 of Story 3 (how openly each developer builds its models). It reads the December 2025 edition of the Stanford Foundation Model Transparency Index, joins each of the 100 indicators to its domain and subdomain, and writes one row for each developer and indicator.

**Question served.** Story 3: "how openly each developer builds its models and how often it appears in reported incidents, so that I can tell the committee what is known and unknown about the developers we would rely on."

**How to run.** Run all cells from top to bottom. Nothing needs to be edited. The notebook reuses the repository file already in `raw/` (see `REFRESH` in Section 3). The last section lists the files to copy by hand into the story's `data/` folder.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one ZIP file (the whole GitHub repository), plus one hand-made table in `manual/` |
| Where it comes from | Stanford Center for Research on Foundation Models. Repository https://github.com/stanford-crfm/fmti |
| Licence and authorisation | Creative Commons Attribution 4.0 (file `LICENSE.md` in the repository). No account or key needed. Details in Section 1 |
| Download size | about 15 MB (the repository holds all three editions and PDFs) |
| Expected time | download 5 to 20 seconds, the whole notebook under 10 seconds with the file in `raw/` |
| Data as of | `processed/data_currency.csv`. The December 2025 scores are fixed for that edition |
| To update | The scores do not change. Read "How to update" at the end for a new edition |

## 1. Source, reference links and authorisation

| Item | Detail |
|---|---|
| Publisher | Stanford Center for Research on Foundation Models (CRFM) and partners |
| Dataset | Foundation Model Transparency Index, December 2025 edition. 100 indicators, 13 developers |
| Repository | https://github.com/stanford-crfm/fmti |
| File used | https://github.com/stanford-crfm/fmti/archive/refs/heads/main.zip, files `fmti-main/Dec2025/Dec2025_scores.csv` and `fmti-main/Dec2025/Dec2025_indicators.csv` |
| Licence | Creative Commons Attribution 4.0 (`LICENSE.md` in the ZIP). Credit Stanford CRFM |
| Authorisation | No account, key or login needed. The file is public. We do not scrape any page |
| Method note | The repository README says that 7 developers agreed to submit their own transparency report and that the Index team prepared reports for 6 of the others. Two researchers score each developer on each indicator, 1 for disclosed and 0 for not disclosed |
| Credit line for the page | Data: Stanford Foundation Model Transparency Index, December 2025, CC BY 4.0 |

## 2. Table structure

### 2.1 Source files in the ZIP

| File | Structure |
|---|---|
| `Dec2025_scores.csv` | 100 rows (one for each indicator) and 14 columns: `Indicator` and one column for each of the 13 developers. Cells hold 1 (disclosed) or 0 (not disclosed) |
| `Dec2025_indicators.csv` | 100 rows and 6 columns: `Domain` (Upstream, Model or Downstream), `Subdomain`, `Indicator`, `Definition`, `Notes`, `Example disclosure` |

### 2.2 Hand-made table in `manual/`

`fmti_companies.csv` has one row for each of the 13 developers and the columns `headquarters`, `region` (the grouping used on the page: United States, China, Europe and Israel), `self_submitted_report` (the developer agreed to submit its own report, from the repository README), `flagship_model` (the model the Index assessed) and `in_bank_list` (`yes` when the developer is one of the eleven developers in the bank's list that the app covers).

### 2.3 Outputs

| Output file | Content |
|---|---|
| `processed/fmti_dec2025_scores_tidy.csv` | 1,300 rows (13 developers times 100 indicators) with `Indicator`, `Domain`, `Subdomain`, `Developer`, `Disclosed`. Subdomain names are written with a capital letter on each word, for example "Acceptable Use Policy" |
| `processed/fmti_companies.csv` | The hand-made table, checked against the scores |

## 3. Set up the paths and download helper

The notebook writes the file it downloads to `raw/` and the tables it builds to `processed/`. `get_file()` tries a fresh download only when `REFRESH = True`, then the file already in `raw/`.

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "fmti_scores"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER}. Current folder: {cwd}")

MANUAL = HERE / "manual"                       # files that people prepared or saved by hand
STORY_DATA = HERE.parents[1] / "data"          # the story's own data folder. The notebook never writes here.
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)
print("Story data     :", STORY_DATA, "(copy files here by hand, see the last section)")

HEADERS = {"User-Agent": "A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 300
REFRESH = True  # False: reuse the file already in raw/ (same snapshot as the page). True: try a fresh download first.


def record(source_id: str, url: str, path: Path, how: str, keep_date: bool = False) -> None:
    """Add (or replace) the retrieval-log entry for one raw file. keep_date leaves an entry untouched when the file is unchanged."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    if keep_date and log.get(source_id, {}).get("sha256") == hashlib.sha256(content).hexdigest():
        return  # same file as before: keep the original entry (date, address and how it was obtained)
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")

import io
import zipfile

ZIP_URL = "https://github.com/stanford-crfm/fmti/archive/refs/heads/main.zip"
EDITION = "Dec2025"


def get_file(source_id: str, url: str, filename: str) -> Path:
    path = RAW / filename
    if REFRESH:
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path.write_bytes(response.content)
            record(source_id, url, path, "downloaded")
            return path
        except Exception as error:  # noqa: BLE001 - any failure falls back to the file in raw/
            print(f"Download of {source_id} failed ({error}). Using the file in raw/.")
    assert path.exists(), f"{filename} is not in raw/. Download {url} and save it there."
    record(source_id, url, path, "reused file in raw/", keep_date=True)
    return path

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\notebooks\fmti_scores
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\notebooks\fmti_scores\manual
Story data     : d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\data (copy files here by hand, see the last section)


## 4. Acquire the file

In [2]:
zip_path = get_file("fmti_repository", ZIP_URL, "fmti-main.zip")
log = json.loads(LOG_PATH.read_text(encoding="utf-8"))["fmti_repository"]
print(json.dumps(log, indent=2))

with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
    scores_name = next(n for n in names if n.endswith(f"{EDITION}/{EDITION}_scores.csv"))
    indicators_name = next(n for n in names if n.endswith(f"{EDITION}/{EDITION}_indicators.csv"))
    scores_raw = pd.read_csv(io.BytesIO(archive.read(scores_name)))
    indicators_raw = pd.read_csv(io.BytesIO(archive.read(indicators_name)))
print(scores_name, scores_raw.shape)
print(indicators_name, indicators_raw.shape)

{
  "url": "https://github.com/stanford-crfm/fmti/archive/refs/heads/main.zip",
  "file": "fmti-main.zip",
  "how": "downloaded",
  "retrieved_utc": "2026-11-20T13:08:44Z",
  "bytes": 15145526,
  "sha256": "f7c2e5c6feddf7f24a0bf5866f7cea00694d065d06cec3b071699aaa6eb78b15"
}
fmti-main/Dec2025/Dec2025_scores.csv (100, 14)
fmti-main/Dec2025/Dec2025_indicators.csv (100, 6)


## 5. Inspect

Check the shape, the types, the missing values and the labels before changing anything.

In [3]:
assert list(scores_raw.columns[:1]) == ["Indicator"], "Dec2025_scores.csv changed its structure."
developers = list(scores_raw.columns[1:])
print("Developers:", len(developers), developers)
print("Indicators in scores:", len(scores_raw), "| in indicator file:", len(indicators_raw))
print("Values in the score cells:", sorted(pd.unique(scores_raw[developers].values.ravel())))
print("Missing values in scores:", int(scores_raw.isna().sum().sum()))
print("Indicators not in both files:",
      sorted(set(scores_raw["Indicator"]) ^ set(indicators_raw["Indicator"])))
print()
print(indicators_raw.groupby("Domain", sort=False)["Indicator"].count().to_string())
print()
print("Subdomains as printed in the source (note the spaces and capitals):")
print(indicators_raw["Subdomain"].map(repr).unique().tolist())

Developers: 13 ['AI21 Labs', 'Alibaba', 'Amazon', 'Anthropic', 'DeepSeek', 'Google', 'IBM', 'Meta', 'Midjourney', 'Mistral', 'OpenAI', 'Writer', 'xAI']
Indicators in scores: 100 | in indicator file: 100
Values in the score cells: [0, 1]
Missing values in scores: 0
Indicators not in both files: []

Domain
Upstream      34
Model         30
Downstream    36

Subdomains as printed in the source (note the spaces and capitals):
["'Data Acquisition'", "'Data Processing'", "'Data Properties'", "'Compute'", "'Methods'", "'Other resources'", "'Model information'", "'Model access'", "'Capabilities'", "'Risks'", "'Model Mitigations'", "'Release'", "'Usage data'", "'Impact'", "'Post-deployment monitoring'", "'Model Behavior Policy'", "'Downstream mitigations'", "'Acceptable use policy'", "'Accountability '", "'Accountability'"]


## 6. Transform

The scores are turned from one column for each developer into one row for each developer and indicator, and joined to the domain and subdomain of the indicator. Subdomain names are tidied: spaces at the end are removed and each word starts with a capital letter, because the page prints them as headings.

In [4]:
def tidy_name(text: str) -> str:
    return " ".join(word[:1].upper() + word[1:] for word in str(text).strip().split())


indicators = indicators_raw[["Domain", "Subdomain", "Indicator"]].copy()
indicators["Domain"] = indicators["Domain"].str.strip()
indicators["Subdomain"] = indicators["Subdomain"].map(tidy_name)

tidy = (scores_raw.melt(id_vars="Indicator", var_name="Developer", value_name="Disclosed")
        .merge(indicators, on="Indicator", how="left", validate="many_to_one"))
tidy = tidy[["Indicator", "Domain", "Subdomain", "Developer", "Disclosed"]].reset_index(drop=True)
tidy["Disclosed"] = tidy["Disclosed"].astype(int)
print(tidy.shape)
tidy.head()

(1300, 5)


,Indicator,Domain,Subdomain,Developer,Disclosed
0,Data acquisition methods,Upstream,Data Acquisition,AI21 Labs,1
1,Public datasets,Upstream,Data Acquisition,AI21 Labs,1
2,Crawling,Upstream,Data Acquisition,AI21 Labs,1
3,Usage data used in training,Upstream,Data Acquisition,AI21 Labs,1
4,Notice of usage data used in training,Upstream,Data Acquisition,AI21 Labs,1


## 7. Manual layer: the developer table

The table in `manual/fmti_companies.csv` adds what the scores file does not hold. The notebook checks that it lists exactly the developers of the scores file.

In [5]:
companies = pd.read_csv(MANUAL / "fmti_companies.csv")
expected_cols = ["company", "headquarters", "region", "self_submitted_report", "flagship_model", "in_bank_list"]
assert list(companies.columns) == expected_cols, companies.columns.tolist()
only_in_scores = sorted(set(developers) - set(companies["company"]))
only_in_table = sorted(set(companies["company"]) - set(developers))
print("In the scores but not in the developer table:", only_in_scores)
print("In the developer table but not in the scores:", only_in_table)
assert not only_in_scores and not only_in_table, "Update manual/fmti_companies.csv so that it lists the same developers."
print()
print(companies.groupby("region")["company"].count().to_string())
print("Self-submitted reports:", int(companies["self_submitted_report"].eq("yes").sum()), "(README: 7)")
print("In the bank's list:", int(companies["in_bank_list"].eq("yes").sum()))

In the scores but not in the developer table: []
In the developer table but not in the scores: []

region
China                2
Europe and Israel    2
United States        9
Self-submitted reports: 7 (README: 7)
In the bank's list: 8


## 8. Validate

These checks stop the notebook if the table breaks a rule the page relies on.

In [6]:
problems = []
if tidy.duplicated(["Developer", "Indicator"]).any():
    problems.append("a developer and indicator pair appears twice")
if tidy[["Domain", "Subdomain"]].isna().any().any():
    problems.append("an indicator has no domain or subdomain")
if not set(tidy["Disclosed"]) <= {0, 1}:
    problems.append("a score other than 0 or 1")
if len(tidy) != len(developers) * tidy["Indicator"].nunique():
    problems.append("not every developer has every indicator")
if set(tidy["Domain"]) != {"Upstream", "Model", "Downstream"}:
    problems.append(f"domains are {sorted(set(tidy['Domain']))}")
print("Problems:", problems if problems else "none")
assert not problems, problems

print()
print("Indicators:", tidy["Indicator"].nunique(), "| subdomains:", tidy["Subdomain"].nunique())
print("Indicators by domain:", tidy.drop_duplicates("Indicator")["Domain"].value_counts().to_dict())
totals = tidy.groupby("Developer")["Disclosed"].sum().sort_values(ascending=False)
print()
print("Indicators disclosed (of 100), by developer:")
print(totals.to_string())
print("Mean:", round(totals.mean(), 1))

Problems: none

Indicators: 100 | subdomains: 19
Indicators by domain: {'Downstream': 36, 'Upstream': 34, 'Model': 30}

Indicators disclosed (of 100), by developer:
Developer
IBM           95
Writer        72
AI21 Labs     66
Anthropic     46
Google        41
Amazon        39
OpenAI        35
DeepSeek      32
Meta          31
Alibaba       26
Mistral       18
Midjourney    14
xAI           14
Mean: 40.7


## 9. Save

In [7]:
tidy.to_csv(OUT / "fmti_dec2025_scores_tidy.csv", index=False, encoding="utf-8")
companies.to_csv(OUT / "fmti_companies.csv", index=False, encoding="utf-8")
print("saved", OUT / "fmti_dec2025_scores_tidy.csv", len(tidy), "rows")
print("saved", OUT / "fmti_companies.csv", len(companies), "rows")

saved d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\notebooks\fmti_scores\processed\fmti_dec2025_scores_tidy.csv 1300 rows
saved d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\notebooks\fmti_scores\processed\fmti_companies.csv 13 rows


## How to update, and what to do if the download is blocked

**The December 2025 scores do not change.** A new download gives the same scores. The notebook is built to be run again for a new edition of the Index.

**Automatic.** Set `REFRESH = True` in Section 3 and run all cells. Section 4 downloads the repository ZIP and replaces `raw/fmti-main.zip`. About 20 seconds.

**Manual download (use this if the run stops with `Download of fmti_repository failed`).**

1. Open https://github.com/stanford-crfm/fmti in a browser, choose Code and then Download ZIP.
2. Save the file as `raw\fmti-main.zip` (inside this folder), replacing the old file. Keep the name.
3. Run the notebook again.

**New edition.** Change `EDITION` in Section 3 to the folder name of the new edition (for example `Dec2026`). Check that the developers and indicators match the structure in Section 2. Update `manual/fmti_companies.csv` for the new developers, and expect the page text that quotes 13 developers and 100 indicators to need an update.

**Update check.** Open `raw/retrieval_log.json` and `processed/data_currency.csv`.

## Data currency: how up to date is this data?

In [8]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": covers_from,
        "data_covers_to": covers_to,
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


write_currency(
    "Stanford FMTI December 2025 (fmti_dec2025_scores_tidy.csv)",
    "2025-12", "2025-12",
    f"One edition of the index (December 2025). File in raw/ ({log['how']}, retrieved {log['retrieved_utc']}, see raw/retrieval_log.json). "
    "Scores are fixed for that edition and do not change when the repository is downloaded again",
    "First edition October 2023, second May 2024, third December 2025 (13 developers). The repository README gives the history",
    "Irregular, roughly one edition per year")

dataset                  Stanford FMTI December 2025 (fmti_dec2025_scores_tidy.csv)
extracted_utc            2026-11-20T13:08:44Z
data_covers_from         2025-12
data_covers_to           2025-12
snapshot_note            One edition of the index (December 2025). File in raw/ (downloaded, retrieved 2026-11-20T13:08:44Z, see raw/retrieval_log.json). Scores are fixed for that edition and do not change when the repository is downloaded again
source_origin_note       First edition October 2023, second May 2024, third December 2025 (13 developers). The repository README gives the history
source_update_frequency  Irregular, roughly one edition per year


In [9]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 4.2 seconds
Files written:
  processed\data_currency.csv  (598 bytes)
  processed\fmti_companies.csv  (652 bytes)
  processed\fmti_dec2025_scores_tidy.csv  (77,894 bytes)


## 10. Copy the tables into the story data folder (by hand)

The notebook does not write into the story's `data/` folder. After a run, copy the files listed below from `processed/` into `../../data/` by hand. The page reads only those copies. The cell below only prints. It shows which files to copy, their number of rows and whether the copy already in `data/` has the same content.

In [10]:
TO_COPY = ["fmti_dec2025_scores_tidy.csv", "fmti_companies.csv"]   # data_currency.csv stays here as a record of the run
for name in TO_COPY:
    made = OUT / name
    rows_made = len(pd.read_csv(made))
    kept = STORY_DATA / name
    if not kept.exists():
        state = "not in the story data folder yet. Copy it."
    elif pd.read_csv(kept, keep_default_na=False).equals(pd.read_csv(made, keep_default_na=False)):
        state = "the copy in the story data folder has the same content. Nothing to do."
    else:
        state = "the copy in the story data folder is DIFFERENT. Copy the new file over it."
    print(f"{name}: {rows_made} rows. From {made.relative_to(HERE)} to {kept}. Status: {state}")

fmti_dec2025_scores_tidy.csv: 1300 rows. From processed\fmti_dec2025_scores_tidy.csv to d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\data\fmti_dec2025_scores_tidy.csv. Status: the copy in the story data folder has the same content. Nothing to do.
fmti_companies.csv: 13 rows. From processed\fmti_companies.csv to d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\3_reported_incidents\data\fmti_companies.csv. Status: the copy in the story data folder has the same content. Nothing to do.


## 11. Run time and limits

**Time to extract.** One ZIP of about 15 MB. Expect 5 to 20 seconds for the download and under 5 seconds for the rest. The last cells print the measured time.

**Limits and known risks.**

1. The Index scores what a developer discloses publicly about its flagship model (one model for each developer). A high score means more is disclosed, not that the model is safer.
2. A 0 means the Index team did not find the disclosure. It does not say that the developer withholds the information.
3. Seven developers submitted their own reports and the Index team prepared the reports for six. The method differs between the two groups.
4. Z.ai, Moonshot and MiniMax are among the bank's eleven developers but are not scored in the December 2025 edition.
5. Only the December 2025 edition is used. The earlier editions have different indicator lists and cannot be compared directly.

## 12. References

1. Stanford CRFM, Foundation Model Transparency Index repository. https://github.com/stanford-crfm/fmti
2. Bommasani et al., The Foundation Model Transparency Index. https://arxiv.org/abs/2310.12941
3. Creative Commons, Attribution 4.0 International. https://creativecommons.org/licenses/by/4.0/